# Ovoz orqali jinsni aniqlash (o'zbekcha Common Voice)

In [ ]:
# Import sozlamalari
import os       
import time
import csv

import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

# ---- Sozlamalar ----
BASE = r"E:\common voice\uz"          # Common Voice o'zbekcha papkasi
CLIPS = os.path.join(BASE, "clips")    # mp3 fayllar
SR = 22050                             # treningda ham, bashoratda ham BIR XIL sample rate
N_MELS = 128                           # xususiyat vektori uzunligi
FEATURES_DIR = f"features_sr{SR}"      # har bir audio uchun .npy shu yerga saqlanadi
RESULTS_DIR = "results_uz"             # yig'ilgan x, y va model shu yerga
SEED = 42

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# Malumotlarni o'qish
def read_tsv(name):
    return pd.read_csv(
        os.path.join(BASE, f"{name}.tsv"),
        sep="\t",
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip",
        encoding="utf-8",
        low_memory=False,
    )

df = pd.concat([read_tsv("train"), read_tsv("dev")], ignore_index=True)
df = df[["path", "gender"]]
df = df[df["gender"].isin(["male_masculine", "female_feminine"])].reset_index(drop=True)
print(df["gender"].value_counts())

gender
male_masculine     21953
female_feminine    12367
Name: count, dtype: int64


## 3. Muvozanatli dataset (erkak = ayol soni)

In [3]:
male = df[df["gender"] == "male_masculine"]
female = df[df["gender"] == "female_feminine"]

n = min(len(male), len(female))
balanced_df = pd.concat([
    male.sample(n=n, random_state=SEED),      # tasodifiy tanlash, birinchi n tasi emas
    female.sample(n=n, random_state=SEED),
], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 1 = erkak, 0 = ayol
balanced_df["label"] = (balanced_df["gender"] == "male_masculine").astype(int)
balanced_df.to_csv("common_voice_uz_balanced.csv", index=False)

print(balanced_df.shape)
print(balanced_df["gender"].value_counts())

(24734, 3)
gender
male_masculine     12367
female_feminine    12367
Name: count, dtype: int64


## 4. Xususiyat ajratish funksiyasi
Bu funksiya treningda ham, Gradio'da ham ishlatiladi, shuning uchun natijalar mos keladi.

In [4]:
def extract_features(file_name, sr=SR, n_mels=N_MELS):
    x, sample_rate = librosa.load(file_name, sr=sr)
    mel = librosa.feature.melspectrogram(y=x, sr=sample_rate, n_mels=n_mels)
    return np.mean(mel.T, axis=0)

## 5. Barcha audiolardan xususiyat ajratish

In [5]:
failed = []
for name in tqdm(balanced_df["path"], desc="Xususiyat ajratilmoqda"):
    out = os.path.join(FEATURES_DIR, os.path.splitext(name)[0] + ".npy")
    if os.path.exists(out):
        continue
    src = os.path.join(CLIPS, name)
    try:
        np.save(out, extract_features(src))
    except Exception as e:
        failed.append((name, str(e)))

print("Xato bo'lgan fayllar:", len(failed))

Xususiyat ajratilmoqda:   0%|          | 0/24734 [00:00<?, ?it/s]c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xususiyat ajratilmoqda: 100%|██████████| 24734/24734 [34:19<00:00, 12.01it/s]  


Xato bo'lgan fayllar: 0


## 6. x va y ni yig'ish va saqlash

In [6]:
features, labels = [], []
for name, label in zip(balanced_df["path"], balanced_df["label"]):
    f = os.path.join(FEATURES_DIR, os.path.splitext(name)[0] + ".npy")
    if os.path.exists(f):
        features.append(np.load(f))
        labels.append(label)

x_all = np.array(features, dtype=np.float32)
y_all = np.array(labels, dtype=np.float32)
np.save(os.path.join(RESULTS_DIR, "features.npy"), x_all)
np.save(os.path.join(RESULTS_DIR, "labels.npy"), y_all)
print(x_all.shape, y_all.shape)

(24734, 128) (24734,)


## 7. Ma'lumotni yuklash

In [7]:
def load_data(results_dir=RESULTS_DIR):
    features_path = os.path.join(results_dir, "features.npy")
    labels_path = os.path.join(results_dir, "labels.npy")

    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        print("features.npy yoki labels.npy topilmadi. Avval 5- va 6-kataklarni ishga tushiring.")
        return None, None

    return np.load(features_path), np.load(labels_path)

x, y = load_data()
print(x.shape, y.shape)

(24734, 128) (24734,)


## 8. Train / validation / test ga ajratish

In [8]:
from sklearn.model_selection import train_test_split

def split_data(x, y, test_size=0.1, valid_size=0.1):
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=SEED, stratify=y)
    x_train, x_valid, y_train, y_valid = train_test_split(
        x_train, y_train, test_size=valid_size, random_state=SEED, stratify=y_train)
    return {"x_train": x_train, "x_valid": x_valid, "x_test": x_test,
            "y_train": y_train, "y_valid": y_valid, "y_test": y_test}

data = split_data(x, y)
for k, v in data.items():
    print(k, v.shape)

x_train (20034, 128)
x_valid (2226, 128)
x_test (2474, 128)
y_train (20034,)
y_valid (2226,)
y_test (2474,)


## 9. Model

In [9]:
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Input(shape=(N_MELS,)),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),   # 0 = ayol, 1 = erkak
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156,545 (611.50 KB)

 Trainable params: 156,545 (611.50 KB)

 Non-trainable params: 0 (0.00 B)

## 10. O'qitish (EarlyStopping bilan)

In [10]:
MODEL_PATH = os.path.join(RESULTS_DIR, "model.keras")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
]

history = model.fit(
    data["x_train"], data["y_train"],
    batch_size=64, epochs=100,
    validation_data=(data["x_valid"], data["y_valid"]),
    callbacks=callbacks,
)

Epoch 1/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9309 - loss: 0.2349 - val_accuracy: 0.9753 - val_loss: 0.0756
Epoch 2/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9741 - loss: 0.0961 - val_accuracy: 0.9807 - val_loss: 0.0545
Epoch 3/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9799 - loss: 0.0670 - val_accuracy: 0.9825 - val_loss: 0.0431
Epoch 4/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9818 - loss: 0.0557 - val_accuracy: 0.9888 - val_loss: 0.0361
Epoch 5/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9886 - loss: 0.0414 - val_accuracy: 0.9924 - val_loss: 0.0251
Epoch 6/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9881 - loss: 0.0359 - val_accuracy: 0.9919 - val_loss: 0.0241
Epoch 7/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9900 - loss: 0.0361 - val_accuracy: 0.9928 - val_loss: 0.0299
Epoch 8/100
314/314 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9907 - loss: 0.0326 - 

## 11. Test to'plamida baholash

In [11]:
test_loss, test_acc = model.evaluate(data["x_test"], data["y_test"], verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc*100:.2f}%")

from sklearn.metrics import classification_report, confusion_matrix
y_pred = (model.predict(data["x_test"], verbose=0).ravel() > 0.5).astype(int)
print(confusion_matrix(data["y_test"], y_pred))
print(classification_report(data["y_test"], y_pred, target_names=["female", "male"]))

Test loss: 0.0255 | Test accuracy: 99.47%
[[1224   13]
 [   0 1237]]
              precision    recall  f1-score   support

      female       1.00      0.99      0.99      1237
        male       0.99      1.00      0.99      1237

    accuracy                           0.99      2474
   macro avg       0.99      0.99      0.99      2474
weighted avg       0.99      0.99      0.99      2474



## 12. Saqlash

In [12]:
model.save(MODEL_PATH)
print("Saqlandi:", MODEL_PATH)

Saqlandi: results_uz\model.keras


## 13. Bashorat funksiyasi

In [ ]:
def classify_gender(file):
    if file is None:
        return {"error": "Audio fayl tanlanmadi"}

    audio_length = librosa.get_duration(path=file)

    start = time.time()
    features = extract_features(file).reshape(1, -1)
    male_prob = float(model.predict(features, verbose=0)[0][0])
    female_prob = 1 - male_prob
    gender = "male" if male_prob > 0.5 else "female"
    elapsed = time.time() - start

    return {
        "gender": gender,
        "Male_probability": f"{male_prob*100:.2f}%",
        "Female_probability": f"{female_prob*100:.2f}%",
        "Time taken": f"{elapsed:.2f} seconds",
        "Audio length": f"{audio_length:.2f} seconds",
    }

## 14. Gradio interfeysi

In [ ]:
import gradio as gr 

try:
    demo.close()   # oldingi interfeys ochiq bo'lsa, yopamiz
except NameError:
    pass

demo = gr.Interface(
    fn=classify_gender,
    inputs=gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio yuklang yoki yozing"),
    outputs="json",
    title="Ovoz orqali jinsni aniqlash",
)
demo.launch(share=True )

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://d57d072b100919d204.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
